# NZ Retail Sales — Modelling & Evaluation

Evaluates trained LightGBM, Ridge, and Prophet models side-by-side.

In [ ]:
import sys
sys.path.insert(0, '../src')

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from forecasting.config import load_config, resolve_path
from forecasting.data import ADEClient
from forecasting.evaluate import directional_accuracy, mae, mape, residual_analysis, rmse, plot_forecast_vs_actual
from forecasting.features import TimeSeriesFeatureEngineer

cfg = load_config()

## 1. Load Model Bundle

In [ ]:
bundle = joblib.load(resolve_path('models') / 'best_model.joblib')
model = bundle['model']
fe = bundle['feature_engineer']
feature_cols = bundle['feature_cols']
print('Run ID:', bundle['run_id'])
print('Training metrics:', bundle['metrics'])

## 2. Reconstruct Test Split

In [ ]:
client = ADEClient()
merged = client.build_merged_dataset(cfg['data']['start_date'])
featured = fe.transform(merged)

test_months = cfg['model']['test_size_months']
train_df = featured.iloc[:-test_months]
test_df = featured.iloc[-test_months:]

X_test = test_df[feature_cols].values
y_test = test_df['retail_sales'].values

print(f'Test set: {len(test_df)} months ({test_df["date"].min().date()} → {test_df["date"].max().date()})')

## 3. Forecast vs Actual

In [ ]:
preds = model.predict(X_test)
sigma = bundle['metrics'].get('lgbm_rmse', np.std(y_test - preds)) * 1.28

fig = plot_forecast_vs_actual(
    test_df['date'], y_test, preds,
    lower=preds - sigma, upper=preds + sigma,
    title='LightGBM — Test Set Forecast vs Actual (80% PI)'
)
plt.show()

## 4. Metrics Summary

In [ ]:
metrics_summary = pd.DataFrame([{
    'Model': 'LightGBM',
    'RMSE': rmse(y_test, preds),
    'MAE': mae(y_test, preds),
    'MAPE (%)': mape(y_test, preds),
    'Directional Acc.': directional_accuracy(y_test, preds),
}])
metrics_summary.round(3)

## 5. Residual Analysis (Ljung-Box)

In [ ]:
residuals = y_test - preds
lb_result = residual_analysis(residuals, lags=10)
print('Verdict:', lb_result['verdict'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(test_df['date'], residuals, marker='o', markersize=3)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals over Time')
axes[1].hist(residuals, bins=15, edgecolor='white')
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.show()

## 6. Feature Importance (Top 20)

In [ ]:
import lightgbm as lgb
lgb.plot_importance(model, max_num_features=20, figsize=(10, 6))
plt.title('LightGBM Feature Importance (gain)')
plt.tight_layout()
plt.show()